# 04 – Wochenblick

Die letzten 8 Wochen bis zum Stichtag: Umfang, Belastung, Effizienz und Erholung.

**Ablauf jede Woche:**

1. Im Terminal `uv run myfitness update` ausführen. Das lädt neue Daten und baut die Tabellen neu.
2. Hier „Run All“.

Den Stichtag in der nächsten Zelle ändern, um auf eine frühere Woche zurückzublicken. Wochen laufen von Montag bis Sonntag. Die Woche des Stichtags zählt nur bis zum Stichtag.

In [ ]:
from datetime import date, timedelta

REFERENCE_DATE = date.today()
WEEKS = 8

In [ ]:
import matplotlib.pyplot as plt
import polars as pl

from myfitness import plots, week
from myfitness.config import PROCESSED

window = week.window(REFERENCE_DATE, WEEKS)
days_in_current_week = REFERENCE_DATE.weekday() + 1
print(f"Zeitraum {window.first_day:%d.%m.%Y} bis {window.reference_date:%d.%m.%Y}")
print(f"Aktuelle Woche: {days_in_current_week} von 7 Tagen")

## Auf einen Blick

Die Woche des Stichtags im Vergleich zum Mittel der vier Wochen davor. Ist die aktuelle Woche noch nicht vorbei, fallen Summen wie km, Stunden und TRIMP naturgemäß niedriger aus.

In [ ]:
week.summary(window)

## Umfang je Sportart

In [ ]:
plots.plot_weekly_volume(week.weekly_volume(window))
plt.show()

## Belastung

TRIMP pro Tag, Fitness (CTL) und Ermüdung (ATL), Form (TSB). Details in `03_belastung`.

In [ ]:
plots.plot_training_load(week.load_in_window(window), "Belastung – letzte Wochen")
plt.show()

week.weekly_load(window)

## Effizienz

EF der Läufe ab 30 Minuten im Zeitraum. Das 28-Tage-Mittel ist über die ganze Historie gerechnet. Details in `01_effizienz`.

In [ ]:
activities = pl.read_parquet(PROCESSED / "activities.parquet")
activity_metrics = pl.read_parquet(PROCESSED / "activity_metrics.parquet")

ef_trend = plots.ef_trend_table(activity_metrics, activities).filter(
    pl.col("start_local").dt.date().is_between(window.first_day, window.reference_date)
)
if ef_trend.height > 0:
    plots.plot_ef_over_time(ef_trend)
    plt.show()
else:
    print("Keine Läufe ab 30 Minuten im Zeitraum.")

## Erholung und Gewicht

Graue Punkte zeigen die Tageswerte, die blaue Linie das Wochenmittel. Das Gewicht wird zwischen Messungen fortgeschrieben.

In [ ]:
plots.plot_recovery(week.recovery_days(window), week.weekly_recovery(window))
plt.show()

week.weekly_recovery(window)

## Läufe der aktuellen Woche

In [ ]:
current_week_start = REFERENCE_DATE - timedelta(days=REFERENCE_DATE.weekday())
(
    activities.filter(pl.col("start_local").dt.date().is_between(current_week_start, REFERENCE_DATE))
    .join(activity_metrics.select("activity_id", "trimp", "ef"), on="activity_id", how="left")
    .select(
        "start_local",
        "sport",
        "name",
        km=pl.col("distance_m") / 1000,
        minutes=pl.col("duration_s") / 60,
        avg_hr="avg_hr",
        trimp="trimp",
        ef="ef",
        temp_c="temp_c",
    )
    .sort("start_local")
)